In [ ]:
# First Order Logic Solver applied to Wumpus World
# Jada Zorn, Sara Croghan

puzzle_level = 'hard'
file_path= f'/content/drive/MyDrive/Caves/{puzzle_level}/path_h4.txt'

## KEY ASSUMPTIONS:
# all variables are lowercase
# all predicates start with an uppercase
# all rules/clauses are disjuncts (made up of liteals OR-ed together)


In [ ]:
# CLASS DEFINTIONS

# parse term for unification algorithm (changed to handle skolem functions)
def parse_term(term):
  # split up a term into its predicate and argument string, can handle skolem functions (toy problem)
  # input: term, (literal)
  # output: list of components of the input term
  # if ( then it must be a function or literal
  if '(' in term:
    predicate, argument_string  = term.split('(', 1)
    arguments = argument_string[:-1].split(',')
    parsed_arguments = [parse_term(arg.strip()) for arg in arguments]
    return [predicate] + parsed_arguments
  else:
    return term

# figure out if a string is a variable or not
# input: string in question
# output: true or false
def is_variable(variable):
  return isinstance (variable, str) and variable.islower()


# literal class where each has a sign true = positive, false= negative, a predicate and a list of variables/arguments
class Literal:
  def __init__(self, sign, predicate, variables):
    self.sign = sign # boolean, true for positive false for negative
    self.predicate = predicate # string
    self.variables = variables # list of strs
    term_string = f"{self.predicate}({', '.join(self.variables)})"
    self.parsed_term = parse_term(term_string) # to handle skolem functions

  def negate(self):
    # negate the literal, copy list of variables
    return Literal(not self.sign, self.predicate, list(self.variables))

    # apply subsitutions found from dictionary
    # input: substitution dictionary
    # output: literal with any found subs made
    # nested functions so that apply_subs can be applied recursively but parsing is still done after no matter what
  def substitute(self, substitutions):
    def apply_subs(term):
      # must be a variable to be subbed
      if is_variable(term) and term in substitutions:
        return substitutions[term]
      elif isinstance(term, list):
        # if it is a list iterate through the list and appy sub to each recursively
        return [apply_subs(item) for item in term]
      else:
        return term
        #handle skolem function if necessary, seperate out multiple terms
    new_parsed_term = apply_subs(self.parsed_term)
  # identify predicate after sub
    new_predicate = new_parsed_term[0]
    # id variables after sub
    new_variables = [str(arg).replace("'", "") for arg in new_parsed_term[1:]]

    return Literal(self.sign, new_predicate, new_variables)


# input: two literals
# output: true or false
  def check_complements(self, other):
    # return true if two literals can resolve (they have the same predicate but opposite signs), otherwise false
    if self.predicate == other.predicate and self.sign != other.sign:
      return True
    else:
      return False

  # define how to print
  def __str__(self):
    # if sign is true, sign is positive or false and negative
    sign = "" if self.sign else "¬"
    # create literal in the form Breeze(1,2)
    return f"{sign}{self.predicate}({', '.join(self.variables)})"

# RULE CLASS
class Rule:
  # assuming a rule is just a disjunct of literals (OR-ed together)
  def __init__(self, literals):
    self.literals = literals # list of Literal objects

# for each literal, if a substiution exists for a variable in the list of subsitutions, apply it
# input: subsitution dictionary
# output
  def substitute(self, substitutions):
    new_literals = []
    # go through each literal and change it (subsitute) as necessary, and assemble list of these changed/new literals
    for literal in self.literals:
      # calls on subsitute method from Literal Class (not recursive here)
      new_literal = literal.substitute(substitutions)
      new_literals.append(new_literal)
    return Rule(new_literals)


  #subsumption: identify if one rule is a more general or equal to another rule
  #input: other rule (rule object)
  # output: true (subsumption works) false (fails)
  #surface level check for optimization
  def subsumption(self, other_rule):
    # checks if all literals in one rule are present in another rule
    # get string of literals of other rule for comparison
    other_literals_str = {str(l) for l in other_rule.literals}
    # check if EVERY literal in self is also in other rule
    for self_literal in self.literals:
      if str(self_literal) not in other_literals_str:
        return False # found literal in self that is not in other, subsumption fails
    return True # all literals in self were also in other, so subsumption works

# check if the clause is empty which would mean contradiction
  def is_empty(self):
    return len(self.literals) == 0

# define how to iterate: over the list of literals
  def __iter__(self):
    return iter(self.literals)

#define how to print
  def __str__(self):
    if self.is_empty():
      return "empty"
    return " ∨ ".join(str(literal) for literal in self.literals)

# Knowledge Base, just a fancier list tailored to our needs
class KnowledgeBase:
  def __init__(self, rules= None):
    self.rules = rules if rules else []
    # holds rule objects in a list

  def add_rule(self, rule):
    self.rules.append(rule)

# make facts clauses so that they can be resolved
# input: literal
# output: rule
  def add_fact(self, literal):
    self.add_rule(Rule([literal]))

# define how to iterate
  def __iter__(self):
    return iter(self.rules)
# define how to find kb length
  def __len__(self):
    return len(self.rules)
# define how to retrieve item from kb
  def __getitem__(self, index):
    return self.rules[index]
# define how to print
  def __str__(self): # might need to fix
    return "\n".join(str(rule) for rule in self.rules)

In [ ]:
# Load in Cave file

#only for importing files from google drive --DELETE BEFORE TURN IN
from google.colab import drive
drive.mount('/content/drive')


# function to load in cave file
# input: file path, knowledge base object
# output: knowledge base (KB object), number of arrows (int), safe query (literal), unsafe query (literal),
# solution (string), grid size (assumed square: int), x query coordiante (str), y query coordinate (str)
def load_cave_file(file_path, knowledge_base):
  arrows= None
  query_safe = None
  query_unsafe = None
  solution = None
  query_coordinates = None
  grid_size = None


  #keep track of if we're reading path, query, resolution, etc
  reading_state = None
# access file, get lines
  with open(file_path, 'r') as file:
    lines = file.readlines()

    for line in lines:
      line = line.strip()
      if not line:
        continue

      # extract grid size
      if line.startswith('GRID:'):
        size_str = line.split(':')[1].strip().lower()
        grid_size = int(size_str.split('x')[0])
        reading_state = None

      # extract num of arrows
      elif  line.startswith('ARROWS:'):
        arrows = int(line.split(':')[1].strip())
        reading_state = None
# change state to reading path
      elif line.startswith('PATH:'):
        reading_state = "PATH"
# read in query if on same line, else look to next line (not completely necessary anymore)
      elif line.startswith('QUERY:'):
        parts = line.split(':')
        # Check if coordinates are on the same line
        if len(parts) > 1 and parts[1].strip():
            coord_str = parts[1].strip()
            coordinates = coord_str.strip("()").split(",")
            q_x = coordinates[0].strip()
            q_y = coordinates[1].strip()
            query_coordinates = (q_x, q_y)
            reading_state = None
        else:
            # if query coords are on next line, change state to look for them
            reading_state = "QUERY"

      # read in resolution if it is on the same line, or check next line
      elif line.startswith('RESOLUTION:'):
        parts = line.split(':')
          # check if solution is on same line
        if len(parts) > 1 and parts[1].strip():
            solution = parts[1].strip().upper()
            reading_state = None #continue on
        else:
            # solution is on next line
            reading_state = "GET_RESOLUTION"

      # load in resolution if on next line
      elif reading_state == "GET_RESOLUTION":
        solution = line.upper()
        reading_state = None

      #extract coordiantes if the query is on next line (no longer necessary)
      elif reading_state == "QUERY":
        coordinates = line.strip("()").split(",")
        x = coordinates[0].strip()
        y = coordinates[1].strip()
        query_coordinates = (x,y)
        reading_state = None

# READING IN PATH FACTS
      elif reading_state == "PATH":
        parts = line.split()
        coordinates = parts[0].strip("()").split(",")
        x,y = int(coordinates[0]), int(coordinates[1]) # probably don't need to change these to int

        #Breeze
        breeze_info = parts[1].split(":")[1]
        # check if breeze is detected or not
        breeze_present=(breeze_info == 'T') # true is breeze is true, false if not
        knowledge_base.add_fact(Literal(breeze_present, "Breeze", [str(x),str(y)])) # add fact to kb

        #checks to make sure that the adjacent cells to the perceived cell are within the puzzle, if yes then add assumption
        # that they could contain a pit, but don't add to kb yet
        if grid_size:
          adjacent_pit_literals = []
          #north
          if y + 1 < grid_size: adjacent_pit_literals.append(Literal(True, "Pit", [str(x), str(y + 1)]))
          #south
          if y - 1 >= 0:        adjacent_pit_literals.append(Literal(True, "Pit", [str(x), str(y - 1)]))
          #east
          if x + 1 < grid_size: adjacent_pit_literals.append(Literal(True, "Pit", [str(x + 1), str(y)]))
          # west
          if x - 1 >= 0:        adjacent_pit_literals.append(Literal(True, "Pit", [str(x - 1), str(y)]))

          #create a rule that says that if there is a breeze than one of the adjacent cells (literals created above) has a pit
          #implement rule ¬Breeze(x,y) ∨ Pit(adj_cell1) ∨ Pit(adj_cell2), etc. for all valid adjacent cells
          if breeze_present:
            breeze_rule_literals = [Literal(False, "Breeze", [str(x), str(y)])] + adjacent_pit_literals
            knowledge_base.add_rule(Rule(breeze_rule_literals))
          else:
            # implement rule Breeze(x,y) ∨ ¬Pit(adj_cell) for each valid adjacent cell
            # if there is not a breeze, adjacent cells don't have pit
            # must be seperate rule for each adjacent cell
            for pit_literal in adjacent_pit_literals:
              no_pit_rule = Rule([Literal(True, "Breeze", [str(x), str(y)]), pit_literal.negate()])
              knowledge_base.add_rule(no_pit_rule)

        # extract stench info from other part of the line
        stench_info = parts[2].split(":")[1]
        stench_present=(stench_info == 'T') # true is yes stench false if no stench
        knowledge_base.add_fact(Literal(stench_present, "Stench", [str(x),str(y)])) #add the literal to kb

        #checks to make sure that the adjacent cells to the perceived cell are within the puzzle, if yes then add assumption
        if grid_size:
          # create a list of adjacent cells and their literals (assuming there is a wumpus)
          adjacent_wumpus_literals = []
          #north
          if y + 1 < grid_size: adjacent_wumpus_literals.append(Literal(True, "Wumpus", [str(x), str(y + 1)]))
          #south
          if y - 1 >= 0:        adjacent_wumpus_literals.append(Literal(True, "Wumpus", [str(x), str(y - 1)]))
          # east
          if x + 1 < grid_size: adjacent_wumpus_literals.append(Literal(True, "Wumpus", [str(x + 1), str(y)]))
          #west
          if x - 1 >= 0:        adjacent_wumpus_literals.append(Literal(True, "Wumpus", [str(x - 1), str(y)]))

          if stench_present:
              # if there is a stench then create rule to add to kb because these adj cells could have wumpus
              # ¬Stench(x,y) ∨ Wumpus(adj_Cell1) ∨ Wumpus(adj_Cell2)
              rule_literals = [Literal(False, "Stench", [str(x), str(y)])] + adjacent_wumpus_literals
              knowledge_base.add_rule(Rule(rule_literals))
          else:
            # if there is no stench adjacent cells can't have wumpus so create rule for each adj cell literal
              #  Stench(x,y) ∨ ¬Wumpus(adj_Cell) for each valid adjacent cell, add to kb
              for wumpus_literal in adjacent_wumpus_literals:
                  no_wumpus_rule = Rule([Literal(True, "Stench", [str(x), str(y)]), wumpus_literal.negate()])
                  knowledge_base.add_rule(no_wumpus_rule)

    # extract query coordinates
    x_q,y_q = query_coordinates
    # build safe and unsafe queries (as literals)
    query_safe = Literal(True, "Safe", [x_q, y_q])
    query_unsafe = Literal(True, "Unsafe", [x_q, y_q])

    return knowledge_base, arrows, query_safe, query_unsafe, solution, grid_size, x_q, y_q

Mounted at /content/drive


In [ ]:

# unification

# ASSUMING THAT ALL VARIABLE ARE LOWERCASE AND ALL PREDICATES START WITH AN UPPERCASE
# SUBSTITUTIONS IS A DICTIONARY MAPPING STRING TO STRING

# safety measure to prevent infinite loop, infinite nesting
# checks to see if a variable appears anywhere inside the value its trying to be unified with
def occurs_check(variable, value, substitutions):
  # check if val and var are the same, recursions should end here or false
  # PARAMETERS
  # variable: variable to check
  # value: expression to check
  # subsititutions: current dictionary of variable substitutions
  # RETURNS
  # True if the variable occurs in the value, Fale otherwise
  if variable == value:
    return True
  # if value is a variable and already has a sub, check the sub recursively, handles indirect occurences
  elif is_variable(value) and value in substitutions:
    return occurs_check(variable, substitutions[value], substitutions)
  # if the value is a list repeat check for all its parts and return true if
  # occurs check returns true for any of the value parts if they are the same
  elif isinstance(value, list):
    return any(occurs_check(variable, val, substitutions) for val in value)
  else:
    # doesn't occur
    return False

def unify(exp1, exp2, substitutions= None):
  # PARAMETERS
  # exp1, exp2: teo exptessions to unify
  # substitutions: dictonary od existing variable substitutions
  # RETURNS
  # substitutions: dictionary mapping variables to unified variables, None if unfication fails
  # count: total number of unifcation attempts (int)
  count = 1 # unfiication counter
  # initialize dictionary if needed
  if substitutions is None:
    substitutions = {}
# base case for recursion, if expressions are identical, return
  if exp1 == exp2:
    return substitutions, count
# if expression 1 is a variable pass on to unify variable to handle things at the variable level
  if is_variable(exp1):
    substitutions, new_count = unify_variable(exp1, exp2, substitutions)
    return substitutions, count + new_count
# if expression 2 is a variable pass on to helper function to unify at this level
  if is_variable(exp2):
    substitutions, new_count = unify_variable(exp2, exp1, substitutions)
    return substitutions, count + new_count

# to handle literals and skolem functions (toy problem only)
  if isinstance(exp1, list) and isinstance(exp2, list):
    # if it is two lists we are trying to unify and they dont match size it fails
    if len(exp1) != len(exp2) :
      return None, count
      # recursively go through the list of variables and try and unify if lengths match
    new_substitutions = substitutions
    for item1, item2 in zip(exp1, exp2):# pairs up elements to try and unify (one set then the next)
    # try and unify the subexpression pairs, and pass these subsitutions on
      new_substitutions, recursive_count = unify(item1, item2, new_substitutions)
      count += recursive_count
      # if any recursive call fails, everything fails, this will return none up the chain
      if new_substitutions is None:
        return None, count
    return new_substitutions, count # is subs are found return these

  return None, count #expressions cannot be unified

def unify_variable(var, val, substitutions):
  # check if one can substitute the other and vice versa
  # PARAMETERS
  # var: variable to unify
  # val: expression to unfiy variable with
  # substiutions: dictionary of var to val sub mappings
  # RETURNS
  # new_substiutions: updated subsituions, None if unification fails
  # count: updated unficiation count
  count = 1 # init count for this specific instance
  if var in substitutions: # check if var is already in sub dict and if so make the sub
    substitutions, new_count = unify(substitutions[var], val, substitutions)
    return substitutions, count + new_count
  if is_variable(val) and val in substitutions: # check if val is already in sub dict (meaning it is a varible) and if yes make sub
    substitutions, new_count = unify(var, substitutions[val], substitutions)
    return substitutions, count + new_count
  # check if var already occurs anywhere inside val to avoid infitie loop
  if occurs_check(var, val, substitutions):
    return None, count # fail
  # update substitition list adds a new mapping between a variable and a value
  # if all checks pass then add new map between var and val
  new_substitutions = substitutions.copy()
  new_substitutions[var] = val
  return new_substitutions, count


In [ ]:
# resolution

def resolve(rule1, rule2):
  # Attempts to resolve 2 rules and return a list of new rules
  # PARAMETERS
  # rule1, rule2: 2 rule objects containing a list of litrals
  # RETURNS
  # new_rules: list of new rules resulting from resolution
  # total_unify_count: number of unifcation attempts during resolution (int)

  total_unify_count = 0
  new_rules = []
  for literal1 in rule1:
    for literal2 in rule2:
      if literal1.check_complements(literal2):
        # Attempt to unify the literals
        substitution, unify_count = unify(literal1.parsed_term, literal2.parsed_term)
        total_unify_count += unify_count

        if substitution is not None:
          # Apply substitution to all literals
          literals_new = []

          # Add literals from rule1, besides literal1
          for l in rule1:
            if l != literal1:
              literals_new.append(l.substitute(substitution))

          # Add literals from rule2, besides literal2
          for l in rule2:
            if l != literal2:
              literals_new.append(l.substitute(substitution))

          # Remove duplicates
          no_duplicates_literals = []
          for l in literals_new:
            if str(l) not in [str(x) for x in no_duplicates_literals]:
              no_duplicates_literals.append(l)

          new_rules.append(Rule(no_duplicates_literals))

  return new_rules, total_unify_count

def resolution(kb, query):
  # PARAMETERS
  # kb: KnowledgeBase object containing rules
  # query: query literal to prove during resolution. Will be negated for proof by contradiction
  # RETURNS
  # result: True if the query is provided by kb, False if else
  # resolution_counter: total number of resolution attempts (int)
  # unification_countr: total number of unification attemtps (int)

  # negate query and add to KB
  query_negated = query.negate()
  #put in rule form
  query_negated_rule = Rule([query_negated])

  #initiate set of support so we are only resolving against descendants of the query, goal is to make this empty to
  # find a contradiction
  set_of_support = [query_negated_rule]

  # keep track of which clauses have already been seen to keep from redoing steps, for efficiency
  seen_clauses = {str(rule) for rule in kb.rules}
  seen_clauses.add(str(query_negated_rule))

  # max num of clauses as a safety measure
  max_clauses = 50000
  resolution_counter = 0
  unification_counter = 0

  i=0
#while there is still something is set of support
  while i < len(set_of_support):
    current_rule = set_of_support[i]
    i+= 1 #move to next iteration

    #take a clause from set of support and try and resolve against everyother clause in the KB
    for rule in kb.rules:
      new_rules, unifications = resolve(current_rule, rule)
      resolution_counter += 1
      unification_counter += unifications
      for r in new_rules:
        if r.is_empty():
          #if resolution returns an empty list then contradication was found
          return True, resolution_counter, unification_counter # success
# convert to stirng for better matching
        r_str = str(r)
         #the new clause that comes from the resolution, if it hasn't already been seen, then add it to the set
         # if it passes subsumption
        if r_str not in seen_clauses:
          # initial subsumption boolean
          subsumption = False
          # before adding r as a new rule into the knowledge base, check to see if it passes subsumption with a rule
          # already in the rules we are resolving against
          for rule in kb.rules:
            if rule.subsumption(r):
              subsumption = True
              break # found a rule that subsumes with r so we can stop, don't add it to our kb/seen rules
          # if r is not subsumed and is new in information, add it to seen set and set of support to be resolved against
          # of support so we can perform resolutions on it
          if not subsumption:
            seen_clauses.add(r_str)
            # because these new rules are descended from negated query, add them to set of support
            set_of_support.append(r)

# take same clause (form set of support) and try and resolve it against other clauses in set of support
      for rule in set_of_support[:i-1]:
        new_rules, unifications = resolve(current_rule, rule)
        resolution_counter += 1
        unification_counter += unifications
        for r in new_rules:
          if r.is_empty():
            #if resolution returns an empty list then contradication was found
            return True, resolution_counter, unification_counter # success
            r_str = str(r)
          #the new clause that comes from the resolution, if it hasn't already been seen, then add it to the set
          # if it passes subsumption
          if r_str not in seen_clauses:
            subsumption = False
            for existing_rule in (kb.rules + set_of_support):
              if existing_rule.subsumption(r):
                subsumption = True
                break # found a rule that subsumes with r so we can stop, don't add it to our kb/seen rules
            if not subsumption:
              seen_clauses.add(r_str)
              set_of_support.append(r)

# max number of clauses check,
      if(len(seen_clauses) > max_clauses):
        print("max clauses reached.. uh oh!")
        return False, resolution_counter, unification_counter #fail

  return False, resolution_counter, unification_counter #no contradiction found


In [ ]:
# Output Cave File

def output(final_result, file_path, knowledge_base):
  # PARAMETERS
  # final_result: result of query resolution for test cell (string)
  # file_path: path to the tested cave file (string)
  # knowledge_base: KB object containing rules used in resolution

  # Get path ID from input file name
  file_name = file_path.split("/")[-1]
  name = file_name.split(".")[0]
  path_id = name.split("_")[-1]

  # Create output file name
  output_name = f"group_11_path_{path_id}.txt"

  # Make kb rules a string for easy printing
  kb_string = "\n".join(str(rule) for rule in knowledge_base.rules)

  # Write to file
  with open(output_name, "w") as file:
    file.write("Clauses used:\n")
    file.write(kb_string + "\n\n")
    file.write("QUERY: " + final_result)



In [ ]:
# MAIN CALL BLOCK
# RUN THIS FOR FULL AUTOMATION OF CODE
# UPDATE FILE PATH IN FIRST CODE CELL

from itertools import combinations
kb = KnowledgeBase()

#Add General Wumpus World Rules (Standardized Apart)
kb.add_rule(Rule([Literal(False, "Wumpus", ["x6", "y6"]), Literal(True, "Unsafe", ["x6", "y6"])]))
kb.add_rule(Rule([Literal(False, "Pit", ["x6", "y6"]),    Literal(True, "Unsafe", ["x6", "y6"])]))
kb.add_rule(Rule([Literal(True, "Wumpus", ["x7", "y7"]), Literal(True, "Pit", ["x7", "y7"]), Literal(True, "Safe", ["x7", "y7"])]))

#Load the specific cave file
kb, arrows, query_safe, query_unsafe, solution, grid_size, q_x, q_y = load_cave_file(file_path, kb)
# try to find resolution on basic kb
# query safe
is_safe, res_safe, uni_safe = resolution(kb, query_safe)

# query unsafe
is_unsafe, res_unsafe, uni_unsafe = resolution(kb, query_unsafe)

# determine safe, unsafe, risky
final_result = ""
if is_safe:
    final_result = "SAFE"
elif is_unsafe:
    final_result = "UNSAFE"
else:
    final_result = "RISKY"
# solution check below to keep from searching if answer is already found
if final_result == "RISKY" and solution != final_result: # if answer is inconclusive at first, see what we know about non pit, wumpus and non wumpus locations
  # logic to locate wumpuses, wumpus free cells, and pit free cells and add this info to kb
  print("INCONCLUSIVE RESULTS... searching further")
  stench_locations = set()
  visited_cells = set()
  no_stench_locations = set()
  breeze_locations = set()
  no_breeze_locations = set()

  for rule in kb.rules:
    # look for rules that are just a literal (can tell us if a cell has been visited or has stench)
    if len(rule.literals) ==1:
      literal = rule.literals[0] # extract singular literal
      if literal.predicate in ["Stench", "Breeze"]: # all literals with info about a cell have Stench and Breeze predicates
        x,y = int(literal.variables[0]), int(literal.variables[1])
        visited_cells.add((x,y))

        if literal.predicate == "Stench":
          if literal.sign is True: # this is a stench cell, so add it to stench set
            stench_locations.add((x,y))
          else: # if there is no stench at cell, add it to no stench set
            no_stench_locations.add((x,y))
        if literal.predicate == "Breeze":
          if literal.sign is True: # this is a breeze cell, so add it to breeze set
            breeze_locations.add((x,y))
          else: # if there is no breeze at cell, add it to no breeze set
            no_breeze_locations.add((x,y))

  # determine wumpus free cells by looking a adjacent cells to no stench cells
  wumpus_free_cells = set()
  for (ns_x,ns_y) in no_stench_locations:
    # find north east south west neighbors of no stench cells
    for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
      nw_x, nw_y = ns_x + dx, ns_y + dy # cells that don't have a wumpus because they are adjacent to no stench
      # if the no stench adjacent cell is within the grid and hasn't been visited then we know for sure it doesn't have a wumpus
      if 0 <= nw_x < grid_size and 0 <= nw_y < grid_size and (nw_x, nw_y) not in visited_cells:
        wumpus_free_cells.add((nw_x, nw_y))
        # we can add this fact to our kb
        wumpus_free_rule = Rule([Literal(False, "Wumpus", [str(nw_x), str(nw_y)])])
        kb.add_rule(wumpus_free_rule)

  # determine cells that are PIT FREE
  pit_free_cells = set()
  for (nb_x, nb_y) in no_breeze_locations:
    # find north east south west neighbors of no breeze cells
    for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
      np_x, np_y = nb_x + dx, nb_y + dy # cells that don't have a pit because they are adjacent to no breeze
      if 0 <= np_x < grid_size and 0 <= np_y < grid_size and (np_x, np_y) not in visited_cells: # cells that for sure
      # don't have a pit because they are not adjacent to a breeze and they are within the cave and they haven't been visited
        pit_free_cells.add((np_x, np_y))
        pit_free_rule = Rule([Literal(False, "Pit", [str(np_x), str(np_y)])])
        kb.add_rule(pit_free_rule)

  # determine all candidate cells for a wumpus that are adjacent to a stench cell and in bounds
  wumpus_candidate_cells = set()
  for (stench_x, stench_y) in stench_locations:
    #find all north east , south west adjacent cells of stench cells
    for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
      candidate_x, candidate_y = stench_x + dx, stench_y + dy
      #check to see if this candidate is within grid and HAS NOT been visited if yes, add to candidate set
      if 0 <= candidate_x < grid_size and 0 <= candidate_y < grid_size and (candidate_x, candidate_y) not in visited_cells:
        wumpus_candidate_cells.add((candidate_x, candidate_y))

  wumpus_candidate_cells.difference_update(wumpus_free_cells) # remove wumpus free cells from candidate set

  # out of the group of wumpus candidates, create groups 1 larger than the number of wumpuses
  # so that when looking at a group of wumpus candidates this size we know that 1 candidate must not contain a wumpus
  cell_group_size = arrows +1
  # we can only generate all possible combos of candidates if there are enough candidates to form a group
  if len(wumpus_candidate_cells) >= cell_group_size:
    # generate all possible combinations of candidate cells of size arrow + 1 to create a disjunct rule
    # that will make it so one of the candidate cells MUST NOT contain a wumpus
    for cell_group in combinations(wumpus_candidate_cells, cell_group_size):
      literals = [] # all the literals to go into disjunct
      for cell in cell_group:
        # for each cell create a literal that states there isn't a wumpus
        literals.append(Literal(False, "Wumpus", [str(cell[0]), str(cell[1])]))
      kb.add_rule(Rule(literals)) # add the disjunct of literals (as a rule) into the kb

  elif len(wumpus_candidate_cells) > 0: # if there are the same num of candidates as wumpuses, assume these are the wumpus locations
    for cell in wumpus_candidate_cells:
      kb.add_rule(Rule([Literal(True, "Wumpus", [str(cell[0]), str(cell[1])])]))


  # re run queries on new kb
  # query safe
  is_safe, res_safe_final, uni_safe_final = resolution(kb, query_safe)

  # query unsafe
  is_unsafe, res_unsafe_final, uni_unsafe_final = resolution(kb, query_unsafe)

  res_safe = res_safe + res_safe_final
  res_unsafe = res_unsafe + res_unsafe_final
  uni_safe = uni_safe + uni_safe_final
  uni_unsafe = uni_unsafe + uni_unsafe_final

 # see if anything changed
  if is_safe:
    final_result = "SAFE"
  elif is_unsafe:
    final_result = "UNSAFE"
  # if neither then its still risky

total_resolutions = res_safe + res_unsafe
total_unifications = uni_safe + uni_unsafe

# FOR TESTING
#print(f"Conclusion: The cell is {final_result}")
#print(f"Total Unification Attempts: {total_unifications}")
#print(f"Total Resolution Steps: {total_resolutions}")
#print(f"Expected solution from file: {solution}")

output(final_result, file_path, kb)
